# 🛡️ SPECTRE-GRID — Treinamento Reprodutível e Livre de Vazamento (Google Colab Pro · Tesla T4)

**Arquitetura:** `CNN-1D -> LSTM -> GATConv -> FC Classifier` (`SPECTRE_GRID`)  
**Repositório:** `abraaoteixeira/ids-cnn-lstm-gnn` · **Branch:** `fix/leakage-free-pipeline`  
**Alvo Científico:** IEEE/IFIP NOMS 2027

---

### Protocolo Metodológico Obrigatório
1. **Particionamento Temporal Estrito (60% / 20% / 20%):** Divisão cronológica em Treino, Validação e Teste Reservado **antes** de qualquer transformação estatística.
2. **Zero Data Leakage:** Vocabulário One-Hot, seleção das Top-20 features por correlação de Pearson e `StandardScaler` ajustados **exclusivamente no conjunto de treino**.
3. **Snapshots Temporais Discretos $G_t = (V_t, E_t)$:** Grafos construídos por janelas de 30s e 60s sem atravessar fronteiras de partição, sem arestas futuras e com rotulagem restrita ao emissor malicioso na janela.
4. **Escopo Compatível com eBPF (`--feature_scope ebpf_compatible`):** Exclui atributos de inspeção profunda de payload L7/pós-conexão indisponíveis no plano de dados.
5. **Tolerância a Falhas no Colab (`--resume`):** Salvamento periódico de estado a cada época diretamente no Google Drive, permitindo retomada imediata em caso de desconexão da sessão.

### Divisão de Trabalho Paralelo (Dell G15 RTX 3050 vs. Colab Pro Tesla T4)
| Ambiente | GPU / VRAM | Tarefa Designada | Janela Temporal | Sementes (`--seed`) |
| :--- | :--- | :--- | :---: | :---: |
| **Dell G15 (Local/Tailscale)** | RTX 3050 (6 GB) | Snapshots de **60s** (`CNN-LSTM-GAT` vs. `CNN-LSTM`) | `60s` | `42, 1337, 2027` |
| **Google Colab Pro (Nuvem)** | Tesla T4 (15 GB) | Snapshots de **30s** (maior densidade de grafos, `CNN-LSTM-GAT` vs. `CNN-LSTM`) + Dataset Completo | `30s` / `60s` | `42, 1337, 2027` |

In [ ]:
# 1. Verificação de Hardware e GPU CUDA (Tesla T4 15GB)
!nvidia-smi

import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device name:     {torch.cuda.get_device_name(0)}")
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f"VRAM total:      {vram_gb:.2f} GB")
else:
    raise RuntimeError("GPU CUDA não detectada! Em 'Ambiente de execução -> Alterar o tipo de ambiente de execução', selecione GPU (T4).")

In [ ]:
# 2. Montagem do Google Drive e Criação da Estrutura Persistente
import os
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT = Path('/content/drive/MyDrive/SPECTRE_GRID_NOMS2027')
except ImportError:
    print('Executando fora do Google Colab; usando diretório local ./colab_drive_mirror')
    DRIVE_ROOT = Path('./colab_drive_mirror').resolve()

DIRS = {
    'datasets': DRIVE_ROOT / 'datasets',
    'snapshots': DRIVE_ROOT / 'snapshots',
    'checkpoints': DRIVE_ROOT / 'checkpoints',
    'reports': DRIVE_ROOT / 'reports',
    'figures': DRIVE_ROOT / 'figures',
}
for name, path in DIRS.items():
    path.mkdir(parents=True, exist_ok=True)
    print(f"[OK] {name:12s} -> {path}")

In [ ]:
# 3. Clone do Repositório na Branch Isolada `fix/leakage-free-pipeline` e Instalação do PyG
import os

REPO_DIR = '/content/ids-cnn-lstm-gnn' if os.path.exists('/content') else os.path.abspath('.')
if REPO_DIR == '/content/ids-cnn-lstm-gnn':
    if not os.path.exists(REPO_DIR):
        !git clone https://github.com/abraaoteixeira/ids-cnn-lstm-gnn.git {REPO_DIR}
    os.chdir(REPO_DIR)
    !git fetch origin
    !git checkout -B fix/leakage-free-pipeline origin/fix/leakage-free-pipeline
    !git status --short --branch

!pip install -q torch_geometric pandas numpy scikit-learn pytest matplotlib seaborn

In [ ]:
# 4. Portão Obrigatório de Integridade Científica (8 Testes Anti-Vazamento + GATConv em CUDA)
!python -m pytest tests/test_leakage_free_pipeline.py -v
!python model.py

In [ ]:
# 5. Ingestão Segura do Dataset UNSW-NB15 a partir do Google Drive
# Faça upload de `UNSW-NB15.csv.gz` (40 MB) ou `UNSW-NB15.csv` (161 MB) para:
# Meu Drive/SPECTRE_GRID_NOMS2027/datasets/
import gzip
import hashlib
import shutil
from pathlib import Path

local_csv = Path('data/raw/benchmarks/UNSW-NB15.csv')
local_csv.parent.mkdir(parents=True, exist_ok=True)

drive_csv = DIRS['datasets'] / 'UNSW-NB15.csv'
drive_gz = DIRS['datasets'] / 'UNSW-NB15.csv.gz'

if local_csv.exists() and local_csv.stat().st_size > 1_000_000:
    print(f'Dataset local já presente: {local_csv} ({local_csv.stat().st_size / (1024**2):.2f} MB)')
elif drive_csv.exists() and drive_csv.stat().st_size > 1_000_000:
    print(f'Copiando {drive_csv} do Google Drive para o armazenamento rápido da VM...')
    shutil.copy2(drive_csv, local_csv)
elif drive_gz.exists():
    print(f'Descompactando {drive_gz} do Google Drive para {local_csv}...')
    with gzip.open(drive_gz, 'rb') as f_in, open(local_csv, 'wb') as f_out:
        shutil.copyfileobj(f_in, f_out)
else:
    print('AVISO: UNSW-NB15.csv(.gz) ainda não encontrado no Drive. Gerando amostra sintética determinística para validação funcional.')

if local_csv.exists() and local_csv.stat().st_size > 0:
    sha256 = hashlib.sha256(local_csv.read_bytes()).hexdigest()[:16]
    print(f'Arquivo pronto: {local_csv} | Tamanho: {local_csv.stat().st_size / (1024**2):.2f} MB | SHA256 prefix: {sha256}')

In [ ]:
# 6. Construção e Persistência dos Snapshots Temporais G_t (30s e 60s) no Google Drive
from pathlib import Path

local_csv = Path('data/raw/benchmarks/UNSW-NB15.csv')
snap_30s_pt = DIRS['snapshots'] / 'unsw_snapshots_30s.pt'
snap_60s_pt = DIRS['snapshots'] / 'unsw_snapshots_60s.pt'
topk_json = DIRS['reports'] / 'top20_ebpf_clean.json'

# Ajuste MAX_ROWS=None para processar todas as 700.000 linhas na T4 (15 GB VRAM)
MAX_ROWS = 200000
max_rows_flag = f'--max_rows {MAX_ROWS}' if MAX_ROWS else ''

if local_csv.exists() and local_csv.stat().st_size > 1_000_000:
    if not snap_30s_pt.exists():
        !python preprocessor.py --input_csv {local_csv} {max_rows_flag} --window_sec 30 --feature_scope ebpf_compatible --output_pt {snap_30s_pt} --topk_json {topk_json}
    if not snap_60s_pt.exists():
        !python preprocessor.py --input_csv {local_csv} {max_rows_flag} --window_sec 60 --feature_scope ebpf_compatible --output_pt {snap_60s_pt} --topk_json {topk_json}
else:
    !python preprocessor.py --window_sec 30 --feature_scope ebpf_compatible --output_pt {snap_30s_pt} --topk_json {topk_json}
    !python preprocessor.py --window_sec 60 --feature_scope ebpf_compatible --output_pt {snap_60s_pt} --topk_json {topk_json}

In [ ]:
# 7. Execução dos Experimentos no Colab Pro (T4) com Salvamento Periódico e Retomada Automática (--resume)
# Caso a sessão do Colab caia, basta reexecutar esta célula: os experimentos concluídos são pulados
# e o experimento em andamento retoma automaticamente da última época salva no Google Drive!
import subprocess

EXPERIMENTS = [
    # (window_label, data_pt, seed, model_variant)
    ('30s', DIRS['snapshots'] / 'unsw_snapshots_30s.pt', 123,  'cnn_lstm_pure'),
    ('30s', DIRS['snapshots'] / 'unsw_snapshots_30s.pt', 123,  'gat_self_loop'),
    ('30s', DIRS['snapshots'] / 'unsw_snapshots_30s.pt', 123,  'cnn_lstm_gat'),
    ('60s', DIRS['snapshots'] / 'unsw_snapshots_60s.pt', 123,  'cnn_lstm_pure'),
    ('60s', DIRS['snapshots'] / 'unsw_snapshots_60s.pt', 123,  'gat_self_loop'),
    ('60s', DIRS['snapshots'] / 'unsw_snapshots_60s.pt', 123,  'cnn_lstm_gat'),
]

EPOCHS = 30
BATCH_SIZE = 8
PATIENCE = 7
VAL_ONLY = True

for win_label, data_pt, seed, variant in EXPERIMENTS:
    tag = f'colab_{variant}_win{win_label}_seed{seed}'
    ckpt_out = DIRS['checkpoints'] / f'{tag}_best.pth'
    resume_out = DIRS['checkpoints'] / f'{tag}_resume.pt'
    report_out = DIRS['reports'] / f'{tag}_report.json'
    cm_out = DIRS['figures'] / f'{tag}_val_cm.png'

    if report_out.exists():
        print(f'[SKIP] Experimento {tag} já finalizado em {report_out}')
        continue

    cmd = [
        'python', 'train.py',
        '--data_path', str(data_pt),
        '--model_variant', variant,
        '--epochs', str(EPOCHS),
        '--batch_size', str(BATCH_SIZE),
        '--patience', str(PATIENCE),
        '--seed', str(seed),
        '--device', 'cuda',
        '--checkpoint_out', str(ckpt_out),
        '--resume_state_out', str(resume_out),
        '--resume',
        '--save_every', '1',
        '--report_out', str(report_out),
        '--cm_out', str(cm_out),
    ]
    if VAL_ONLY:
        cmd.append('--val_only')

    print(f'\n================ EXECUTANDO: {tag} ================')
    subprocess.run(cmd, check=True)

In [ ]:
# 8. Consolidação das Métricas Científicas (Validação vs. Teste Reservado + Latência)
import json
import pandas as pd

rows = []
for rep_file in sorted(DIRS['reports'].glob('*_report.json')):
    with open(rep_file, 'r', encoding='utf-8') as f:
        r = json.load(f)
    test_m = r.get('held_out_test_metrics') or {}
    val_m = r.get('validation_metrics') or {}
    lat = r.get('validation_latency') or r.get('inference_latency') or {}
    cm = val_m.get('confusion_matrix') or test_m.get('confusion_matrix') or {}
    rows.append({
        'Report': rep_file.stem,
        'Variant': r.get('model_variant'),
        'Seed': r.get('seed'),
        'Best Epoch': r.get('best_epoch_on_validation'),
        'Val Prec': val_m.get('precision'),
        'Val Rec': val_m.get('recall'),
        'Val F1': val_m.get('f1_score'),
        'Val FPR': val_m.get('fpr'),
        'Val TN': cm.get('tn'),
        'Val FP': cm.get('fp'),
        'Val FN': cm.get('fn'),
        'Val TP': cm.get('tp'),
        'Train Wall Time (s)': r.get('training_wall_time_sec'),
        'Latency (ms/snap)': lat.get('ms_per_snapshot'),
        'Latency (us/node)': lat.get('us_per_node'),
    })

if rows:
    df_res = pd.DataFrame(rows)
    summary_csv = DIRS['reports'] / 'consolidated_noms2027_metrics.csv'
    df_res.to_csv(summary_csv, index=False)
    display(df_res)
    print(f'\nResumo consolidado salvo em: {summary_csv}')
else:
    print('Nenhum relatório encontrado ainda em', DIRS['reports'])